# CVE Dataset and Its Re-exploration in the UCKG Project

- Name: Yejun Na
- Email: yna1@angelo.edu

## Project Description

The Common Vulnerabilities and Exposures (CVE) dataset, maintained by MITRE, is a publicly available catalog that documents known cybersecurity vulnerabilities in software and hardware systems. Each CVE entry represents a standardized identifier for a specific vulnerability, providing a structured and consistent way to reference and share vulnerability data across tools, organizations, and security processes. CVE entries are derived from reports by researchers, vendors, and coordination centers, and include vital information such as:

- CVE ID (referenceable globally)
- Description of the vulnerability
- Affected vendors and products
- CVSS severity scores and vectors (v2/v3)
- Publication and last modified dates
- References to advisories, reports, and patches
- Weakness mappings (CVE to CWE)
- Configurations (CPEs)
- Exploitability and impact metrics
- Source of record (e.g., CNA)

The CVE dataset is closely integrated with related resources such as the National Vulnerability Database (NVD), which enriches CVE records with additional metadata including CVSS scores, affected configurations, and references. This structured vulnerability information supports a range of cybersecurity tasks including vulnerability management, threat intelligence, patch prioritization, and risk assessment.

As part of the Unified Cybersecurity Knowledge Graph (UCKG) project, we re-explore and integrate CVE data to establish semantically rich relationships across multiple cybersecurity knowledge domains. By mapping CVE entries to related entities such as CWEs, CPEs, and attack techniques from MITRE ATT&CK, the UCKG provides a unified, queryable structure that enhances understanding of how documented vulnerabilities align with system configurations and adversarial behaviors. This integration supports automated security analysis, proactive risk mitigation, and the development of intelligent, graph-based defense strategies across the cyber threat landscape.

## Project Setup

In [31]:
from neo4j import GraphDatabase
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

uri = "bolt://localhost:7687"
username = "neo4j"
password = "abcd90909090"

driver = GraphDatabase.driver(uri, auth=(username, password))

def run_query(query, parameters=None):
    try:
        with driver.session() as session:
            result = session.run(query, parameters)
            return [record.data() for record in result]
    except Exception as e:
        print(f"Error running query: {e}")
        return []

query = """
MATCH (n:UcoCVE)
RETURN 'UcoCVE' AS UcoCVE,
       COUNT(DISTINCT n) AS Total
"""

params = {}
records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df


,UcoCVE,Total
0,UcoCVE,292488


| **Label** | **Total Records** |
| --------- | ----------------- |
| `UcoCVE`  | **292,488**       |


In [30]:
def run_query(query, parameters=None):
    try:
        with driver.session() as session:
            result = session.run(query, parameters)
            return [record.data() for record in result]
    except Exception as e:
        print(f"Error running query: {e}")
        return []

def get_df(query, query_name):
    records = run_query(query)
    if records:
        df = pd.DataFrame(records)
        print(f"{query_name} query result:")
        print(df.head())
        print(f"Total {df.shape[0]} records loaded.\n")
        return df
    else:
        print(f"No results found for {query_name} query.")
        return pd.DataFrame()

outgoing_rel = """
MATCH (n:UcoCVE)-[r]->(m)
RETURN labels(n) AS NodeLabels, labels(m) AS ConnectedNodeLabels, type(r) AS Relationship
"""

incoming_rel = """
MATCH (n:UcoCVE)<-[r]-(m)
RETURN labels(n) AS NodeLabels, labels(m) AS ConnectedNodeLabels, type(r) AS Relationship
"""

undirected_rel = """
MATCH (n:UcoCVE)-[r]-(m)
RETURN labels(n) AS NodeLabels, labels(m) AS ConnectedNodeLabels, type(r) AS Relationship
"""

df_outgoing = get_df(outgoing_rel, "Outgoing Relationships")
df_incoming = get_df(incoming_rel, "Incoming Relationships")
df_undirected = get_df(undirected_rel, "Undirected Relationships")

Outgoing Relationships query result:
           NodeLabels   ConnectedNodeLabels Relationship
0  [Resource, UcoCVE]  [Resource, UcoexCPE]  UCOEXHASCPE
1  [Resource, UcoCVE]  [Resource, UcoexCPE]  UCOEXHASCPE
2  [Resource, UcoCVE]  [Resource, UcoexCPE]  UCOEXHASCPE
3  [Resource, UcoCVE]  [Resource, UcoexCPE]  UCOEXHASCPE
4  [Resource, UcoCVE]  [Resource, UcoexCPE]  UCOEXHASCPE
Total 638189 records loaded.

Incoming Relationships query result:
           NodeLabels           ConnectedNodeLabels  Relationship
0  [Resource, UcoCVE]  [Resource, UcoVulnerability]  UCOHASCVE_ID
1  [Resource, UcoCVE]  [Resource, UcoVulnerability]  UCOHASCVE_ID
2  [Resource, UcoCVE]  [Resource, UcoVulnerability]  UCOHASCVE_ID
3  [Resource, UcoCVE]  [Resource, UcoVulnerability]  UCOHASCVE_ID
4  [Resource, UcoCVE]  [Resource, UcoVulnerability]  UCOHASCVE_ID
Total 292488 records loaded.

Undirected Relationships query result:
           NodeLabels           ConnectedNodeLabels  Relationship
0  [Resource, UcoCVE]  

|   Node   | Relationship Type | Direction  | Connected To       | Count   |
| ---------| ----------------- | ---------- | ------------------ | ------- |
| `UcoCVE` | UCOEXHASCPE       | Outgoing   | `UcoexCPE`         | 638,189 |
| `UcoCVE` | UCOHASCVE\_ID     | Incoming   | `UcoVulnerability` | 292,488 |
| `UcoCVE` | (Both)            | Undirected | Mixed              | 930,677 |


In [22]:
query = """
MATCH (n:UcoExploitTarget)-[r:UCOHASVULNERABILITY]->(m:UcoVulnerability)
RETURN 'UcoExploitTarget' AS SourceNode, 'UCOHASVULNERABILITY' AS Relationship,
        'UcoVulnerability' AS TargetNode, 
        COUNT(DISTINCT n) AS Source_Total, 
       COUNT(DISTINCT r) AS Relationship_Total, 
       COUNT(DISTINCT m) AS Target_Total
UNION
MATCH (n:UcoVulnerability)-[r:UCOHASCVE_ID]->(m:UcoCVE)
RETURN 'UcoVulnerability' AS SourceNode, 'UCOHASCVE_ID' AS Relationship,
        'UcoCVE' AS TargetNode, 
       COUNT(DISTINCT n) AS Source_Total, 
       COUNT(DISTINCT r) AS Relationship_Total, 
       COUNT(DISTINCT m) AS Target_Total
UNION
MATCH (n:UcoCVE)-[r:UCOEXHASCPE]->(m:UcoexCPE)
RETURN 'UcoCVE' AS SourceNode, 'UCOEXHASCPE' AS Relationship,
        'UcoexCPE' AS TargetNode, 
       COUNT(DISTINCT n) AS Source_Total, 
       COUNT(DISTINCT r) AS Relationship_Total, 
       COUNT(DISTINCT m) AS Target_Total
"""

params = {}
records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df


,SourceNode,Relationship,TargetNode,Source_Total,Relationship_Total,Target_Total
0,UcoExploitTarget,UCOHASVULNERABILITY,UcoVulnerability,688,226017,208382
1,UcoVulnerability,UCOHASCVE_ID,UcoCVE,292488,292488,292488
2,UcoCVE,UCOEXHASCPE,UcoexCPE,250205,638189,135326


| **Source Node**    | **Relationship**      | **Target Node**    | **# Source Nodes** | **# Relationships** | **# Target Nodes** |
| ------------------ | --------------------- | ------------------ | ------------------ | ------------------- | ------------------ |
| `UcoExploitTarget` | `UCOHASVULNERABILITY` | `UcoVulnerability` | 688                | 226,017             | 208,382            |
| `UcoVulnerability` | `UCOHASCVE_ID`        | `UcoCVE`           | 292,488            | 292,488             | 292,488            |
| `UcoCVE`           | `UCOEXHASCPE`         | `UcoexCPE`         | 250,025            | 638,189             | 135,326            |


| Metric                       | Count     |
| ---------------------------- | --------- |
| Total unique CVEs            | 292,488   |
| Total affected CPEs          | 135,326   |
| Total mapped vulnerabilities | 292,488   |
| Total exploit targets        | 688       |
| Total relationships (all)    | 1,156,694 |


In [23]:
query = """
MATCH (n:UcoCVE)
UNWIND keys(n) AS Properties
RETURN DISTINCT Properties
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,Properties
0,ucoevaluatorSolution
1,uri
2,ucouserInteractionRequired
3,ucobaseSeverity
4,ucovulnStatus
5,label
6,ucoobtainAllPrivilege
7,ucovectorString
8,ucoexploitabilityScore
9,ucoimpactScore


| **Property Name**            | **Description**                                                                       |
| ---------------------------- | ------------------------------------------------------------------------------------- |
| `ucoevaluatorSolution`       | Description of solutions or fixes (e.g., patches, workarounds) for the CVE.           |
| `uri`                        | Unique URI identifier (may be used as the resource's RDF URI in linked data).         |
| `ucouserInteractionRequired` | Boolean indicating if user action is required to exploit the vulnerability.           |
| `ucobaseSeverity`            | Qualitative severity rating (e.g., "Low", "Medium", "High") based on CVSS.            |
| `ucovulnStatus`              | Status of the CVE in the disclosure process (e.g., "Analyzed", "Deferred").           |
| `label`                      | General label for the node (could be `CVE-yyyy-xxxx` identifier or node label).       |
| `ucoobtainAllPrivilege`      | Boolean; `true` if exploiting the CVE gives full system/administrator access.         |
| `ucovectorString`            | CVSS vector string (e.g., `AV:N/AC:L/PR:N/...`) describing how the score was derived. |
| `ucoexploitabilityScore`     | Numeric score (0.0–10.0) representing how easy it is to exploit the CVE.              |
| `ucoimpactScore`             | Numeric score (0.0–10.0) representing the potential damage if exploited.              |


In [24]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucobaseSeverity AS Severity, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,Severity,Count
0,[MEDIUM],108716
1,[HIGH],58123
2,[LOW],19662
3,[],105987


What is a CVE Severity Score?
- The severity score refers to the CVSS (Common Vulnerability Scoring System) score, which quantifies the seriousness of a vulnerability. 
- This score helps organizations prioritize which vulnerabilities to address first.

| **Severity** | **CVSS Score Range** | **Meaning**                          | **Count** |
| ------------ | -------------------- | ------------------------------------ | --------- |
| `HIGH`       | 7.0 – 10.0           | Severe impact, needs urgent fix      | 58,123    |
| `MEDIUM`     | 4.0 – 6.9            | Moderate impact, needs review        | 108,716   |
| `LOW`        | 0.1 – 3.9            | Limited impact, often lower priority | 19,662    |
| *(Missing)*  | N/A                  | Severity not assigned                | 105,987   |


In [25]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucovulnStatus AS Status, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,Status,Count
0,[Deferred],94597
1,[Rejected],15081
2,[Undergoing Analysis],921
3,[Analyzed],21958
4,[Modified],134873
5,[Awaiting Analysis],24862
6,[Received],196


The ucovulnStatus (or vulStatus) property represents the current lifecycle or review state of a CVE record within the National Vulnerability Database (NVD) or the CVE publication pipeline. It's essentially a status label used by MITRE and NIST to indicate where a vulnerability stands in terms of review, analysis, or publication.
| **Status**            | **Count** | **Meaning**                                                            |
| --------------------- | --------- | ---------------------------------------------------------------------- |
| `Modified`            | 134,873   | The CVE has been reviewed and updated, usually with improved metadata. |
| `Deferred`            | 94,597    | The CVE has been postponed or withheld from public disclosure.         |
| `Analyzed`            | 21,958    | The CVE has been reviewed and validated by analysts.                   |
| `Rejected`            | 15,081    | The CVE ID was issued but later ruled invalid.                         |
| `Awaiting Analysis`   | 24,862    | The report was received but not yet reviewed by analysts.              |
| `Undergoing Analysis` | 921       | The CVE is currently being assessed before publication.                |
| `Received`            | 196       | The CVE was submitted and is waiting in the queue for triage.          |



In [26]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucoexploitabilityScore AS ExploitabilityScore, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,ExploitabilityScore,Count
0,[10.0],65991
1,[3.9],21429
2,[1.9],661
3,[4.9],3863
4,[],105987
5,[8.6],54063
6,[3.4],4086
7,[8.0],20631
8,[6.8],9396
9,[3.1],591


The ExploitabilityScore is a subscore from the CVSS (Common Vulnerability Scoring System) that estimates how easy it is for an attacker to exploit a vulnerability. It’s part of the CVSS Base Score, which is the overall severity score.
| **Exploitability Score Range** | **Severity Category** | **Representative Scores in Data**         | **Total Count** |
| ------------------------------ | --------------------- | ----------------------------------------- | --------------- |
| `9.0 – 10.0`                   | Very Easy to Exploit  | 10.0, 8.6, 8.0                            | 140,685         |
| `6.0 – 8.9`                    | Easy to Exploit       | 6.8, 6.5, 6.4                             | 11,270          |
| `4.0 – 5.9`                    | Moderate Effort       | 5.5, 5.1, 4.9, 4.4, 4.1                   | 6,313           |
| `1.0 – 3.9`                    | Hard to Exploit       | 3.9, 3.4, 3.2, 3.1, 2.7, 2.5, 2.2, 2.0... | 26,345          |
| `0.0`                          | Not Exploitable       | 0.0                                       | 0 (not shown)   |
| *(Missing)*                    | Unknown               | `[]`                                      | 105,987         |



In [27]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucoimpactScore AS ImpactScore, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,ImpactScore,Count
0,[2.9],79135
1,[10.0],30652
2,[6.4],58462
3,[4.9],9628
4,[],105987
5,[6.9],7397
6,[0.0],22
7,[9.2],495
8,[8.5],405
9,[7.8],256


The ImpactScore is a CVSS subscore that quantifies the potential impact of a vulnerability on the confidentiality, integrity, and availability of a system. It represents how much damage the vulnerability could cause if successfully exploited.
| **Impact Score Range** | **Severity Category** | **Representative Scores in Data** | **Total Count** |
| ---------------------- | --------------------- | --------------------------------- | --------------- |
| 9.0 – 10.0             | Critical Impact       | 10.0, 9.5, 9.2                    | 31,196          |
| 6.0 – 8.9              | High Impact           | 8.5, 7.8, 6.9, 6.4                | 66,859          |
| 4.0 – 5.9              | Moderate Impact       | 4.9                               | 9,628           |
| 1.0 – 3.9              | Low Impact            | 2.9                               | 79,135          |
| 0.0                    | No Impact             | 0.0                               | 22              |
| *(Missing)*            | Unknown               | `[]`                              | 105,987         |



In [28]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucouserInteractionRequired AS InteractionRequired, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,InteractionRequired,Count
0,[false],237375
1,[true],55113


The ucouserInteractionRequired (or simply User Interaction Required) field in a CVE record indicates whether a successful exploitation of the vulnerability requires any action by a user — such as clicking a link, opening a file, or interacting with a malicious interface.

| **Interaction Required** | **Meaning**                                                                                      | **Count** |
| ------------------------ | ------------------------------------------------------------------------------------------------ | --------- |
| `false`                  | **No user action is needed** — the exploit can be triggered automatically                        | 237,375   |
| `true`                   | **User action is required** — the attacker needs the victim to interact (e.g., click, open file) | 55,113    |

- Over 81% of vulnerabilities can be exploited without any user interaction — posing higher risk to systems that are unattended or connected to networks.
- ~19% require human interaction, often through social engineering or tricking users into enabling the exploit.

In [29]:
query = """
MATCH (n:UcoCVE)
RETURN n.ucoobtainAllPrivilege AS ObtainAllPrivilege, count(*) AS Count
"""
params = {}

records = run_query(query, parameters=params)

df = pd.DataFrame(records)

df

,ObtainAllPrivilege,Count
0,[false],287263
1,[true],5225


The ucobtainAllPrivilege (or ObtainAllPrivilege) property indicates whether a vulnerability can allow an attacker to gain full or system-level privileges — essentially, complete control over the affected system or application.

| **ObtainAllPrivilege** | **Meaning**                                                                                       | **Count** |
| ---------------------- | ------------------------------------------------------------------------------------------------- | --------- |
| `false`                | The vulnerability **does not** allow an attacker to gain **all system privileges**                | 287,263   |
| `true`                 | The vulnerability **does** allow an attacker to obtain **complete privileges** (e.g., root/admin) | 5,225     |



- Only ~1.8% of vulnerabilities result in total system takeover (privilege escalation to full control).
- Majority (~98%) do not lead to complete privilege acquisition — still harmful, but more limited in scope.
